# Notebook 01 - ML Lifecycle and Dataset Exploration

This notebook revises the complete ML lifecycle and connects each step to MLflow terminology.

**Trainer goal:** Students should understand what features, target, preprocessing, train-test split, parameters, metrics, and artifacts mean before learning MLflow.

In [ ]:
# Core libraries used throughout the Day 1 notebooks
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer



## 1. Load the classification dataset

We use a synthetic customer churn dataset. It contains both numerical and categorical columns, which is useful because real ML projects usually contain mixed data types.

In [ ]:
# Load dataset from the local datasets folder
churn_df = pd.read_csv('../datasets/customer_churn_training.csv')

# Show first 5 records
churn_df.head()

In [ ]:
# Check number of rows and columns
# In MLflow, dataset details can later be logged as metadata or artifacts.
print('Rows:', churn_df.shape[0])
print('Columns:', churn_df.shape[1])

In [ ]:
# Understand column data types
# object columns are usually categorical/string columns.
churn_df.info()

In [ ]:
# Basic statistical summary for numerical columns
churn_df.describe().T

In [ ]:
# Check missing values column-wise
# Missing value handling is a preprocessing step.
# Different preprocessing choices can produce different model results.
churn_df.isna().sum().sort_values(ascending=False)

## 2. Identify features and target

In supervised learning:

- **Features / X**: input columns used for prediction
- **Target / y**: output column we want to predict

For churn prediction, `churn` is the target.

In [ ]:
# We should not use customer_id because it is only an identifier.
# It does not carry meaningful predictive pattern for a new unseen customer.
X = churn_df.drop(columns=['customer_id', 'churn'])
y = churn_df['churn']

print('Feature columns:', list(X.columns))
print('Target column: churn')

In [ ]:
# Separate numerical and categorical columns.
# This is important because they need different preprocessing steps.
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print('Numerical features:', numerical_features)
print('Categorical features:', categorical_features)

## 3. Train-test split

We split the data into:

- **Training data**: used for learning patterns
- **Testing data**: used for checking performance on unseen data

In MLflow, values like `test_size` and `random_state` are good examples of experiment parameters.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,      # 20% data for testing
    random_state=42,    # fixed value for reproducibility
    stratify=y          # keeps churn ratio similar in train and test
)

print('Training rows:', X_train.shape[0])
print('Testing rows:', X_test.shape[0])

## 4. Create a preprocessing pipeline

Real datasets contain:

- Missing numerical values
- Missing categorical values
- Categorical text columns
- Numerical columns with different scales

A pipeline keeps these transformations clean and repeatable.

In [ ]:
# Numerical pipeline:
# 1. Fill missing values using median
# 2. Scale values so algorithms like Logistic Regression work better
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline:
# 1. Fill missing values using most frequent value
# 2. Convert categories into numeric one-hot columns
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, numerical_features),
    ('cat', categorical_pipeline, categorical_features)
])

print('Preprocessing pipeline created successfully.')